# Reinforcement Learning Fundamentals

This notebook introduces the agent–environment loop and tabular Q-learning through a five-position hallway. Run the cells in order, predict outcomes before running code, and answer the short checkpoints in chat.

## 1. The environment

The agent starts at position `0` and wants to reach position `4`. Its actions are `left` and `right`. Each step returns `(next_state, reward, done)`. Reaching the goal gives reward `+1` and ends the episode; other moves give reward `0` and continue.

In [1]:
LEFT = 0
RIGHT = 1
GOAL = 4

def reset():
    # return to state 0, reward 0, done False
    return 0, 0, False


def step(state, action):
    # move left (-1) or right (+1)
    next_state = state - 1 if action == LEFT else state + 1
    # with max 0, position cannot be negative
    # with min GOAL, position cannot exceed 4
    next_state = max(0, min(GOAL, next_state))
    # terminal state when agent reaches goal (position 4)
    done = next_state == GOAL
    # reward only when agent reaches goal (position 4)
    reward = 1 if done else 0
    # return next state, reward, done
    return next_state, reward, done

In [2]:
print('reset():', reset())
# expect: (0, 0, False)
print('step(0, LEFT):', step(0, LEFT))
# expect: (0, 0, False)
print('step(2, RIGHT):', step(2, RIGHT))
# expect: (3, 0, False)
print('step(3, RIGHT):', step(3, RIGHT))
# expect: (4, +1, True)

reset(): (0, 0, False)
step(0, LEFT): (0, 0, False)
step(2, RIGHT): (3, 0, False)
step(3, RIGHT): (4, 1, True)


### Checkpoint 1 — concept

What are the state, action, reward, and terminal condition in this hallway?

## 2. The Q-table

A Q-table stores one value for every state–action pair. There are five states and two actions, so the table has ten values. Initially, every value is `0`: the agent has no learned preference.

In [3]:
q_table = [[0.0, 0.0] for _ in range(GOAL + 1)]
# 0 for all q values for each state and each action (left, right)
# range(GOAL + 1) gives state 0, 1, 2, 3, 4
# left is represented by left entry, right is represented by right entry
# so the q table is q value for each state and each action (left, right)

for state, values in enumerate(q_table):
    print(f'state {state}: left={values[LEFT]}, right={values[RIGHT]}')

state 0: left=0.0, right=0.0
state 1: left=0.0, right=0.0
state 2: left=0.0, right=0.0
state 3: left=0.0, right=0.0
state 4: left=0.0, right=0.0


### Checkpoint 2 — interpretation

If `Q(2, right) = 5` and `Q(2, left) = 1`, which action does a greedy agent choose, and why?

Choose to go right given state 2 because it yields the maximum expected return of 5

## 3. One Q-learning update

For a non-terminal transition, calculate the target first:

`target = reward + gamma * best_future_Q`

Then calculate the TD error and move the old estimate toward the target:

`new_Q = old_Q + alpha * (target - old_Q)`

`gamma` controls how much future rewards matter. `alpha` controls how quickly the estimate changes.

### Checkpoint 3 — calculation

Given `old_Q=4`, `reward=2`, `gamma=0.9`, `best_future_Q=10`, and `alpha=0.1`, calculate the target and the new Q-value.

In [4]:
old_q = 4
reward = 2
gamma = 0.9
best_future_q = 10
alpha = 0.1

target = reward + gamma * best_future_q
td_error = target - old_q
new_q = old_q + alpha * td_error

print('target:', target)
print('TD error:', td_error)
print('new Q-value:', new_q)

target: 11.0
TD error: 7.0
new Q-value: 4.7


In [5]:
import random

def choose_action(q_table, state, epsilon):
    if random.random() < epsilon:
        return random.choice([LEFT, RIGHT])
    return RIGHT if q_table[state][RIGHT] >= q_table[state][LEFT] else LEFT

def train(episodes=500, alpha=0.1, gamma=0.9):
    q_table = [[0.0, 0.0] for _ in range(GOAL + 1)]
    epsilon = 1.0

    for _ in range(episodes):
        state, _, _ = reset()

        while True:
            action = choose_action(q_table, state, epsilon)
            next_state, reward, done = step(state, action)

            if done:
                target = reward
            else:
                target = reward + gamma * max(q_table[next_state])

            q_table[state][action] += alpha * (target - q_table[state][action])
            state = next_state

            if done:
                break

        epsilon = max(0.05, epsilon * 0.99)

    return q_table

In [6]:
random.seed(7)
q_table = train()

for state, values in enumerate(q_table):
    print(f'state {state}: left={values[LEFT]:.3f}, right={values[RIGHT]:.3f}')

state 0: left=0.656, right=0.729
state 1: left=0.656, right=0.810
state 2: left=0.728, right=0.900
state 3: left=0.806, right=1.000
state 4: left=0.000, right=0.000


### Checkpoint 4 — application

Look at the learned table. At position `2`, which action does the greedy policy choose? Does the result make sense for reaching the goal at position `4`?

In [7]:
def evaluate(q_table):
    state, _, _ = reset()
    states = [state]

    while state != GOAL:
        action = choose_action(q_table, state, epsilon=0.0)
        state, _, _ = step(state, action)
        states.append(state)

    return states

print('Greedy path:', evaluate(q_table))

Greedy path: [0, 1, 2, 3, 4]


## Optional exercise

Change one thing at a time and observe the result: the number of training episodes, `gamma`, the exploration decay, or the reward for reaching the goal. Before running each change, write down what you expect to happen.